# Chapter 5: Confidence intervals as the dual of tests

In [1]:
import numpy as np
from scipy.stats import beta as beta_dist
from statsmodels.stats.proportion import proportion_confint
from expkit.inference.binomial import wilson_ci, clopper_pearson_ci
from expkit.inference.bootstrap import bootstrap_ci
from expkit.plot.style import apply_style
apply_style()

## Loop A: same data, five intervals

In [2]:
for k, n in [(6, 10), (60, 100), (600, 1000)]:
    seq = np.concatenate([np.ones(k), np.zeros(n - k)])
    wald = proportion_confint(k, n, method='normal')
    wilson = wilson_ci(k, n)
    cp = clopper_pearson_ci(k, n)
    bs_lo, bs_hi, _ = bootstrap_ci(seq, n_boot=2000, seed=0)
    a, b = 1 + k, 1 + (n - k)
    bayes = (float(beta_dist.ppf(0.025, a, b)), float(beta_dist.ppf(0.975, a, b)))
    print(f'{k}/{n}:')
    for name, iv in [('Wald', wald), ('Wilson', wilson), ('Clopper-Pearson', cp), ('Bootstrap', (bs_lo, bs_hi)), ('Bayesian', bayes)]:
        print(f'    {name:<18} [{iv[0]:.3f}, {iv[1]:.3f}]')

6/10:
    Wald               [0.296, 0.904]
    Wilson             [0.313, 0.832]
    Clopper-Pearson    [0.262, 0.878]
    Bootstrap          [0.300, 0.900]
    Bayesian           [0.308, 0.833]
60/100:
    Wald               [0.504, 0.696]
    Wilson             [0.502, 0.691]
    Clopper-Pearson    [0.497, 0.697]
    Bootstrap          [0.510, 0.690]
    Bayesian           [0.502, 0.691]
600/1000:
    Wald               [0.570, 0.630]
    Wilson             [0.569, 0.630]
    Clopper-Pearson    [0.569, 0.631]
    Bootstrap          [0.568, 0.630]
    Bayesian           [0.569, 0.630]


## Loop B: Wald breaks at boundaries

In [3]:
for k, n in [(0, 10), (10, 10), (0, 100), (100, 100)]:
    print(f'{k}/{n}:  Wald={proportion_confint(k, n, method="normal")}  Wilson={wilson_ci(k, n)}  Clopper-Pearson={clopper_pearson_ci(k, n)}')

0/10:  Wald=(0.0, 0.0)  Wilson=(0.0, 0.27753279986288926)  Clopper-Pearson=(0.0, 0.30849710781876083)
10/10:  Wald=(1.0, 1.0)  Wilson=(0.7224672001371106, 1.0)  Clopper-Pearson=(0.6915028921812392, 1.0)
0/100:  Wald=(0.0, 0.0)  Wilson=(0.0, 0.03699349820698569)  Clopper-Pearson=(0.0, 0.03621669264517642)
100/100:  Wald=(1.0, 1.0)  Wilson=(0.9630065017930143, 1.0)  Clopper-Pearson=(0.9637833073548235, 1.0)


## Loop D: empirical coverage at N=20

In [4]:
n, n_trials = 20, 2000
rng = np.random.default_rng(0)
counts = rng.binomial(n, 0.5, size=n_trials)
covered = {'Wald': 0, 'Wilson': 0, 'Clopper-Pearson': 0, 'Bayesian': 0}
for k in counts:
    for name, m in [('Wald', 'normal'), ('Wilson', 'wilson'), ('Clopper-Pearson', 'beta')]:
        lo, hi = proportion_confint(int(k), n, method=m)
        if lo <= 0.5 <= hi: covered[name] += 1
    a, b = 1 + int(k), 1 + (n - int(k))
    lo = float(beta_dist.ppf(0.025, a, b)); hi = float(beta_dist.ppf(0.975, a, b))
    if lo <= 0.5 <= hi: covered['Bayesian'] += 1
for name, c in covered.items():
    print(f'{name:<18} empirical coverage = {c/n_trials:.3f}')

Wald               empirical coverage = 0.952
Wilson             empirical coverage = 0.952
Clopper-Pearson    empirical coverage = 0.952
Bayesian           empirical coverage = 0.952
